In [36]:
import numpy as np
import json

In [37]:
# Read text and create vocab
with open("../data/processed/nova-zemya-processed.txt", "r", encoding="utf-8") as f:
    text = f.read()

vocab = sorted(set(text))

print(f"Text size: {len(text)}")
print(f"Vocab size: {len(vocab)}")
print(f"Characters: {''.join(vocab)}")

Text size: 756206
Vocab size: 74
Characters: 
 !"#(),-.:;?АБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЮЯабвгдежзийклмнопрстуфхцчшщъыьюя—…


In [38]:
# Map every unique char to an integer and back
char_to_idx = {ch: i for i, ch in enumerate(vocab)}
idx_to_char = {i: ch for i, ch in enumerate(vocab)}

print(char_to_idx)

{'\n': 0, ' ': 1, '!': 2, '"': 3, '#': 4, '(': 5, ')': 6, ',': 7, '-': 8, '.': 9, ':': 10, ';': 11, '?': 12, 'А': 13, 'Б': 14, 'В': 15, 'Г': 16, 'Д': 17, 'Е': 18, 'Ж': 19, 'З': 20, 'И': 21, 'Й': 22, 'К': 23, 'Л': 24, 'М': 25, 'Н': 26, 'О': 27, 'П': 28, 'Р': 29, 'С': 30, 'Т': 31, 'У': 32, 'Ф': 33, 'Х': 34, 'Ц': 35, 'Ч': 36, 'Ш': 37, 'Щ': 38, 'Ю': 39, 'Я': 40, 'а': 41, 'б': 42, 'в': 43, 'г': 44, 'д': 45, 'е': 46, 'ж': 47, 'з': 48, 'и': 49, 'й': 50, 'к': 51, 'л': 52, 'м': 53, 'н': 54, 'о': 55, 'п': 56, 'р': 57, 'с': 58, 'т': 59, 'у': 60, 'ф': 61, 'х': 62, 'ц': 63, 'ч': 64, 'ш': 65, 'щ': 66, 'ъ': 67, 'ы': 68, 'ь': 69, 'ю': 70, 'я': 71, '—': 72, '…': 73}


In [39]:
# Encode corpus / Tokenize
encoded = np.array([char_to_idx[ch] for ch in text], dtype=np.int32)
print(f"Encoded shape: {encoded.shape}")
print(f"First 50 chars: {text[:50]}")
print(f"First 50 tokens: {encoded[:50]}")

Encoded shape: (756206,)
First 50 chars: По гладката, стръмна южна урва на Амбарица — висок
First 50 tokens: [28 55  1 44 52 41 45 51 41 59 41  7  1 58 59 57 67 53 54 41  1 70 47 54
 41  1 60 57 43 41  1 54 41  1 13 53 42 41 57 49 63 41  1 72  1 43 49 58
 55 51]


In [40]:
# Split dataset into train and validation
split = int(len(encoded) * 0.9)
train_data = encoded[:split]
val_data = encoded[split:]

print(f"Full dataset size: {len(encoded)}")
print(f"Train size: {len(train_data)}")
print(f"Validation size: {len(val_data)}")

Full dataset size: 756206
Train size: 680585
Validation size: 75621


In [41]:
# Save to disk
np.save("../data/datasets/train.npy", train_data)
np.save("../data/datasets/val.npy",   val_data)

with open("../data/vocab/vocab.json", "w", encoding="utf-8") as f:
    json.dump({
        "char_to_idx": char_to_idx,
        "idx_to_char": {str(i): ch for i, ch in idx_to_char.items()} # idx_to_char needs string keys for JSON
    }, f, ensure_ascii=False, indent=2)

print("Saved train.npy, val.npy, vocab.json")

Saved train.npy, val.npy, vocab.json


In [42]:
def get_batch(encoded, seq_len, batch_size):
    starts = np.random.randint(0, len(encoded) - seq_len - 1, batch_size)
    x = np.stack([encoded[s : s + seq_len] for s in starts])
    y = np.stack([encoded[s+1 : s + seq_len + 1] for s in starts])
    return x, y

# Test on train and val separately
x, y = get_batch(train_data, seq_len=100, batch_size=4)
print(f"Train batch — x: {x.shape}, y: {y.shape}")

x_v, y_v = get_batch(val_data, seq_len=100, batch_size=4)
print(f"Val batch   — x: {x_v.shape}, y: {y_v.shape}")

# Decode first sample to verify
print("".join(idx_to_char[i] for i in x[0]))
print("".join(idx_to_char[i] for i in y[0]))

Train batch — x: (4, 100), y: (4, 100)
Val batch   — x: (4, 100), y: (4, 100)
сение и злощастие се отзова лошо на Невенкиното добиване. То излезе нещастно и последвано от тежки б
ение и злощастие се отзова лошо на Невенкиното добиване. То излезе нещастно и последвано от тежки бо
